In [ ]:
import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    print(f"Device Count    : {torch.cuda.device_count()}")
else:
    print("Running on CPU runtime.")

In [ ]:
import torch
x= torch.rand(6,6)
print(x)

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
import torch

print("CUDA Available :", torch.cuda.is_available())
print("Device Name    :", torch.cuda.get_device_name(0))

In [ ]:
import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Device Count    : {torch.cuda.device_count()}")
else:
    print("Running on CPU runtime.")

In [ ]:
from logging import root
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)   

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# Inspect a batch:
# Shape of X is [N, C, H, W]:
#   • N = 64 (Batch size: number of images in this batch)
#   • C = 1  (Channels: 1 = Grayscale)
#   • H = 28 (Height: 28 pixels)
#   • W = 28 (Width: 28 pixels)
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt


training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Mapping integer labels to human-readable clothing names
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}

# Create a figure canvas of size 8x8 inches
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3

for i in range(1, cols * rows + 1):
    # Pick a random sample index from the training dataset
    sample_idx = torch.randint(len(training_data), size=(1,)).item()
    img, label = training_data[sample_idx]
    
    # =========================================================================
    # WHAT DOES [1, 28, 28] MEAN?
    # PyTorch image tensors follow the format: [Channels, Height, Width] -> [C, H, W]
    #   • 1  = Color Channels (1 = Grayscale / Black & White; 3 would be RGB color)
    #   • 28 = Height of the image (28 pixels tall)
    #   • 28 = Width of the image (28 pixels wide)
    # =========================================================================
    
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")  # Turn off coordinate ticks/axes
    
    # img.squeeze() removes the dimension of size 1: converts [1, 28, 28] -> [28, 28]
    # Matplotlib needs a 2D shape [28, 28] to render a grayscale image
    plt.imshow(img.squeeze(), cmap="gray")

plt.show()

In [10]:
import torch
import torch.nn as nn 

# -------------------------------------------------------------------------
# 1. SELECT DEVICE (GPU vs CPU)
# Check if a hardware accelerator (like CUDA GPU) is available; if not, use CPU.
# -------------------------------------------------------------------------
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")


# -------------------------------------------------------------------------
# 2. DEFINE THE NEURAL NETWORK
# In PyTorch, models inherit from nn.Module, which gives them layer-tracking
# and GPU-acceleration capabilities.
# -------------------------------------------------------------------------
class NeuralNetwork(nn.Module):
    def __init__(self):
        # Always call super().__init__() to initialize PyTorch's internal machinery
        super().__init__()
        
        # Flattens 2D image [28, 28] into a 1D flat line of 784 numbers (28 * 28 = 784)
        self.flatten = nn.Flatten()
        
        # nn.Sequential is an assembly line: data flows through each layer in order
        self.linear_relu_stack = nn.Sequential(
            # Layer 1: Takes 784 pixel inputs -> outputs 512 learned features
            nn.Linear(28 * 28, 512),
            # Activation: Replaces negative numbers with 0 (allows learning complex patterns)
            nn.ReLU(),
            
            # Layer 2 (Hidden): Takes 512 features -> outputs 512 new features
            nn.Linear(512, 512),
            nn.ReLU(),
            
            # Layer 3 (Output): Takes 512 features -> outputs 10 scores (logits)
            # One score for each clothing category (T-shirt, Trouser, Sneaker, etc.)
            nn.Linear(512, 10)
        )

    # ---------------------------------------------------------------------
    # IMPORTANT: 'forward' must be aligned with '__init__', NOT indented inside it!
    # This defines the data flow when you call model(image).
    # ---------------------------------------------------------------------
    def forward(self, x):
        # Step 1: Flatten image [1, 28, 28] -> [784]
        x = self.flatten(x)
        
        # Step 2: Pass through linear layers and ReLU activations
        logits = self.linear_relu_stack(x)
        
        # Step 3: Return raw prediction scores (logits) for the 10 classes
        return logits


# -------------------------------------------------------------------------
# 3. INSTANTIATE AND MOVE TO DEVICE
# Create the network object and send its weights to GPU (or CPU)
# -------------------------------------------------------------------------
model = NeuralNetwork().to(device)
print(model)

Using device: cuda
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
